# Sección 7: Filtros, limpieza y transformación inicial

## Proyecto: Relación entre resultados ICFES y acceso a internet por municipio en Colombia

**Curso:** Procesamiento de Datos a Gran Escala  
**Universidad:** Pontificia Universidad Javeriana  
**Metodología:** CRISP-DM (avance; el proceso completo se consolida en la entrega 2)  

---

Este cuaderno documenta el pipeline implementado en `transform_clean.py` usando **solo Apache Spark**.

**Contenido del pipeline:**
- Casteo de variables numéricas y normalización de códigos DANE de municipio (`lpad` a 5 dígitos).
- **Filtros:** registros ICFES con `estu_estadoinvestigacion = PUBLICADO`; `no_de_accesos` no negativos; puntajes globales en rango 0–500.
- **Atípicos:** regla IQR (1.5×IQR) sobre `punt_global` (ICFES) y sobre `no_de_accesos` (internet).
- **Imputación (ejemplo):** media para `punt_ingles` (ICFES) y `cobertura_neta` (cobertura educativa).
- **Transformación:** agregación de accesos a internet por municipio–año, join con cobertura educativa y variable `accesos_por_habitante` (`total_accesos / poblaci_n_5_16`).

**Prerrequisitos:** haber generado `data/parquet/` con `data.py` (en desarrollo: `python data.py --sample 5000`).

Ejecutar las celdas con el directorio de trabajo en la **raíz del repositorio** (donde están `data.py` y `transform_clean.py`).

In [ ]:
import os

if not os.path.isdir("data/parquet/icfes"):
    raise SystemExit(
        "Falta data/parquet/icfes. Ejecute antes: python data.py "
        "(o data.py --sample 5000 para pruebas)."
    )

from transform_clean import build_spark_session, run_pipeline

# Desarrollo: 5000 filas por tabla tras leer Parquet. Producción/clúster: None
LIMIT = 5000  # Cambie a None para procesar todo.

spark = build_spark_session()
spark.sparkContext.setLogLevel("ERROR")
try:
    run_pipeline(
        spark=spark,
        limit_rows=LIMIT,
        write_parquet=True,
        verbose=True,
        show_progress=False,
    )
finally:
    spark.stop()

## Ejecución equivalente por línea de comandos

Por defecto la consola muestra solo una **barra de progreso** y un **resumen corto** (sin volcar planes físicos de Spark). Para el detalle anterior (`printSchema`, `show`), use `--verbose`.

```bash
python transform_clean.py
python transform_clean.py --limit-rows 5000
python transform_clean.py --no-write
python transform_clean.py --verbose
python transform_clean.py --no-progress
```

Salida limpia en `data/parquet_clean/` (subcarpetas `icfes`, `internet`, `bachillerato`, `municipio_internet_cobertura`).